<a href="https://colab.research.google.com/github/Nifal123/financial-/blob/main/05_Final_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 5 - Final Report and GitHub Presentation

Author: Your Name
Date: May 2026

What This Notebook Covers
- Combined dashboard with all key charts
- Full performance summary across all strategies
- Professional tearsheet export
- GitHub README generator


In [1]:
!pip install yfinance plotly scipy -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/portfolio_optimizer/'

daily_returns   = pd.read_csv(f'{save_path}daily_returns.csv',
                               index_col=0, parse_dates=True)
asset_metrics   = pd.read_csv(f'{save_path}asset_metrics.csv',
                               index_col=0)
backtest_results= pd.read_csv(f'{save_path}backtest_results.csv',
                               index_col=0)
opt_results     = pd.read_csv(f'{save_path}optimization_results.csv',
                               index_col=0)

ASSET_NAMES    = list(daily_returns.columns)
NUM_ASSETS     = len(ASSET_NAMES)
RISK_FREE_RATE = 0.05
TRADING_DAYS   = 252

print("All data loaded successfully")
print("Asset metrics    :", asset_metrics.shape)
print("Backtest results :", backtest_results.shape)
print("Opt results      :", opt_results.shape)

Mounted at /content/drive
All data loaded successfully
Asset metrics    : (7, 13)
Backtest results : (5, 8)
Opt results      : (4, 3)


In [2]:
# Rebuild monthly returns for all strategies
# We need this to recreate the cumulative return lines

from scipy.optimize import minimize

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns)

def portfolio_sharpe(weights, mean_returns, cov_matrix, rf=RISK_FREE_RATE):
    ret = portfolio_return(weights, mean_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    return (ret - rf) / vol

def get_max_sharpe_weights(returns_data):
    n   = len(returns_data.columns)
    mu  = returns_data.mean() * TRADING_DAYS
    cov = returns_data.cov() * TRADING_DAYS
    def neg_sharpe(w):
        return -portfolio_sharpe(w, mu, cov)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(n))
    x0     = np.array([1/n] * n)
    result = minimize(neg_sharpe, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x if result.success else x0

def get_min_variance_weights(returns_data):
    n   = len(returns_data.columns)
    cov = returns_data.cov() * TRADING_DAYS
    def port_vol(w):
        return portfolio_volatility(w, cov)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(n))
    x0     = np.array([1/n] * n)
    result = minimize(port_vol, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x if result.success else x0

def get_risk_parity_weights(returns_data):
    n   = len(returns_data.columns)
    cov = returns_data.cov() * TRADING_DAYS
    def rp_error(w):
        w      = np.array(w)
        vol    = portfolio_volatility(w, cov)
        mrc    = np.dot(cov, w) / vol
        rc     = w * mrc
        target = vol / n
        return np.sum((rc - target) ** 2)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0.01, 1) for _ in range(n))
    x0     = np.array([1/n] * n)
    result = minimize(rp_error, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'maxiter': 1000, 'ftol': 1e-12})
    w = result.x if result.success else x0
    return w / w.sum()

def run_backtest(daily_returns, strategy='max_sharpe', lookback_months=12):
    monthly   = daily_returns.resample('ME').apply(lambda x: (1+x).prod()-1)
    port_rets = []
    port_dates= []
    for i in range(lookback_months, len(monthly)):
        train = daily_returns[
            (daily_returns.index >= monthly.index[i - lookback_months]) &
            (daily_returns.index <  monthly.index[i])
        ]
        if len(train) < 20:
            continue
        if strategy == 'max_sharpe':
            weights = get_max_sharpe_weights(train)
        elif strategy == 'min_variance':
            weights = get_min_variance_weights(train)
        elif strategy == 'risk_parity':
            weights = get_risk_parity_weights(train)
        else:
            weights = np.array([1/NUM_ASSETS] * NUM_ASSETS)
        month_ret = monthly.iloc[i].values
        port_ret  = np.dot(weights, month_ret)
        port_rets.append(port_ret)
        port_dates.append(monthly.index[i])
    return pd.Series(port_rets, index=port_dates, name=strategy)

print("Rebuilding backtests for final report...")
bt_max_sharpe   = run_backtest(daily_returns, 'max_sharpe')
bt_min_variance = run_backtest(daily_returns, 'min_variance')
bt_risk_parity  = run_backtest(daily_returns, 'risk_parity')
bt_equal_weight = run_backtest(daily_returns, 'equal_weight')

spy = daily_returns['US Equities'].resample('ME').apply(lambda x: (1+x).prod()-1)
tlt = daily_returns['Long Bonds'].resample('ME').apply(lambda x: (1+x).prod()-1)
benchmark_60_40 = (0.60 * spy + 0.40 * tlt)

common_start    = bt_max_sharpe.index[0]
benchmark_60_40 = benchmark_60_40[benchmark_60_40.index >= common_start]
benchmark_60_40 = benchmark_60_40.reindex(bt_max_sharpe.index)

strategies = {
    'Max Sharpe'     : bt_max_sharpe,
    'Min Variance'   : bt_min_variance,
    'Risk Parity'    : bt_risk_parity,
    'Equal Weight'   : bt_equal_weight,
    '60/40 Benchmark': benchmark_60_40
}

print("All strategies rebuilt successfully")

Rebuilding backtests for final report...
All strategies rebuilt successfully


In [3]:
# MASTER DASHBOARD — all key charts in one figure
# This is what you screenshot for LinkedIn and your resume

colors = ['#F44336','#00BCD4','#FF9800','#4CAF50','#9E9E9E']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Growth of $100,000 by Strategy',
        'Risk vs Return by Asset Class',
        'Drawdown from Peak by Strategy',
        'Annual Returns Heatmap by Strategy'
    ),
    vertical_spacing  = 0.14,
    horizontal_spacing= 0.10
)

# ── TOP LEFT: Cumulative Returns ──────────────────────────────
for i, (name, returns) in enumerate(strategies.items()):
    cum = (1 + returns).cumprod() * 100000
    fig.add_trace(go.Scatter(
        x    = cum.index,
        y    = cum.values,
        name = name,
        line = dict(width=2, color=colors[i],
                    dash='dash' if name == '60/40 Benchmark' else 'solid'),
        showlegend = True
    ), row=1, col=1)

# ── TOP RIGHT: Risk vs Return Scatter ─────────────────────────
asset_colors = ['#2196F3','#FF9800','#4CAF50','#9C27B0',
                '#F44336','#00BCD4','#FF5722']

for i, asset in enumerate(asset_metrics.index):
    fig.add_trace(go.Scatter(
        x    = [asset_metrics.loc[asset, 'Ann. Volatility %']],
        y    = [asset_metrics.loc[asset, 'Ann. Return %']],
        mode = 'markers+text',
        name = asset,
        text = [asset],
        textposition = 'top center',
        marker = dict(size=12, color=asset_colors[i]),
        showlegend = False
    ), row=1, col=2)

# ── BOTTOM LEFT: Drawdowns ────────────────────────────────────
for i, (name, returns) in enumerate(strategies.items()):
    cum = (1 + returns).cumprod()
    rm  = cum.cummax()
    dd  = (cum - rm) / rm * 100
    fig.add_trace(go.Scatter(
        x    = dd.index,
        y    = dd.values,
        name = name,
        line = dict(width=2, color=colors[i]),
        showlegend = False
    ), row=2, col=1)

# ── BOTTOM RIGHT: Annual Returns Heatmap ─────────────────────
annual_data = {}
for name, returns in strategies.items():
    annual = returns.resample('YE').apply(
        lambda x: (1+x).prod()-1) * 100
    annual.index = annual.index.year
    annual_data[name] = annual

annual_df = pd.DataFrame(annual_data)

fig.add_trace(go.Heatmap(
    z           = annual_df.T.values,
    x           = annual_df.index.astype(str),
    y           = annual_df.columns,
    colorscale  = 'RdYlGn',
    zmid        = 0,
    text        = annual_df.T.round(1).values,
    texttemplate= '%{text}%',
    showscale   = False
), row=2, col=2)

fig.update_layout(
    title  = dict(
        text='Multi-Asset Portfolio Optimizer — Master Dashboard',
        font=dict(size=20, color='white')
    ),
    height  = 850,
    template= 'plotly_dark',
    legend  = dict(orientation='h', y=-0.08, x=0),
    font    = dict(family='Arial', size=11)
)

fig.show()
print("Master dashboard complete")

Master dashboard complete


In [4]:
# Final tearsheet — the complete performance summary

def backtest_metrics(monthly_returns_series, rf=RISK_FREE_RATE, name='Strategy'):
    r          = monthly_returns_series.dropna()
    monthly_rf = rf / 12
    ann_return = (1 + r).prod() ** (12/len(r)) - 1
    ann_vol    = r.std() * np.sqrt(12)
    sharpe     = (ann_return - rf) / ann_vol if ann_vol != 0 else 0
    downside   = r[r < monthly_rf].std() * np.sqrt(12)
    sortino    = (ann_return - rf) / downside if downside != 0 else 0
    cum        = (1 + r).cumprod()
    rm         = cum.cummax()
    dd         = (cum - rm) / rm
    max_dd     = dd.min()
    calmar     = ann_return / abs(max_dd) if max_dd != 0 else 0
    total_ret  = (1 + r).prod() - 1
    win_rate   = (r > 0).sum() / len(r)
    return {
        'Strategy'       : name,
        'Ann. Return %'  : round(ann_return * 100, 2),
        'Total Return %' : round(total_ret * 100, 2),
        'Volatility %'   : round(ann_vol * 100, 2),
        'Sharpe Ratio'   : round(sharpe, 3),
        'Sortino Ratio'  : round(sortino, 3),
        'Calmar Ratio'   : round(calmar, 3),
        'Max Drawdown %' : round(max_dd * 100, 2),
        'Win Rate %'     : round(win_rate * 100, 1),
    }

tearsheet = pd.DataFrame([
    backtest_metrics(bt_max_sharpe,   name='Max Sharpe'),
    backtest_metrics(bt_min_variance, name='Min Variance'),
    backtest_metrics(bt_risk_parity,  name='Risk Parity'),
    backtest_metrics(bt_equal_weight, name='Equal Weight'),
    backtest_metrics(benchmark_60_40, name='60/40 Benchmark'),
]).set_index('Strategy')

print("")
print("=" * 65)
print("     FINAL PERFORMANCE TEARSHEET")
print("=" * 65)
print(f"  Period      : {common_start.date()} to {bt_max_sharpe.index[-1].date()}")
print(f"  Universe    : 7 Asset Classes")
print(f"  Rebalancing : Monthly Walk-Forward")
print(f"  Benchmark   : 60/40 Portfolio")
print(f"  Risk-Free   : {RISK_FREE_RATE*100}%")
print("=" * 65)
print("")
print(tearsheet.to_string())
print("")
print("=" * 65)

best = tearsheet['Sharpe Ratio'].idxmax()
print(f"\n  Best Risk-Adjusted Strategy : {best}")
print(f"  Sharpe improvement vs 60/40 : ",
      round(tearsheet.loc[best,'Sharpe Ratio'] -
            tearsheet.loc['60/40 Benchmark','Sharpe Ratio'], 3))


     FINAL PERFORMANCE TEARSHEET
  Period      : 2019-01-31 to 2024-12-31
  Universe    : 7 Asset Classes
  Rebalancing : Monthly Walk-Forward
  Benchmark   : 60/40 Portfolio
  Risk-Free   : 5.0%

                 Ann. Return %  Total Return %  Volatility %  Sharpe Ratio  Sortino Ratio  Calmar Ratio  Max Drawdown %  Win Rate %
Strategy                                                                                                                           
Max Sharpe               24.56          273.55         12.66         1.545          2.510         1.379          -17.81        76.4
Min Variance              2.00           12.62          6.07        -0.494         -0.689         0.131          -15.33        59.7
Risk Parity               5.41           37.21          9.09         0.046          0.067         0.315          -17.21        59.7
Equal Weight              7.77           56.65         11.32         0.244          0.332         0.450          -17.27        66.7
60/40 Benc

In [5]:
# Generate your README.md automatically
# Copy this output directly into your GitHub README

best_strategy = tearsheet['Sharpe Ratio'].idxmax()
best_row      = tearsheet.loc[best_strategy]
bench_row     = tearsheet.loc['60/40 Benchmark']
sharpe_improv = round(
    (best_row['Sharpe Ratio'] - bench_row['Sharpe Ratio'])
    / abs(bench_row['Sharpe Ratio']) * 100, 1)

readme = f"""
# Multi-Asset Portfolio Optimizer

> Institutional-grade portfolio construction across 7 asset classes
> using Modern Portfolio Theory, Risk Parity, and Monte Carlo simulation.

## Project Overview

This project implements a complete quantitative portfolio management
framework — from data pipeline to backtesting — replicating the
methodology used by institutional asset managers.

## Asset Universe

| Ticker | Asset Class        |
|--------|--------------------|
| SPY    | US Equities        |
| EFA    | Intl Equities      |
| TLT    | Long-Term Bonds    |
| BND    | Aggregate Bonds    |
| GLD    | Gold               |
| GSG    | Commodities        |
| VNQ    | Real Estate (REITs)|

## Strategies Implemented

- Mean-Variance Optimization (Markowitz 1952)
- Maximum Sharpe Ratio Portfolio
- Minimum Variance Portfolio
- Risk Parity (Equal Risk Contribution)
- Walk-Forward Backtesting with Monthly Rebalancing

## Backtest Results (2019-2024)

| Strategy        | Ann. Return | Volatility | Sharpe | Max Drawdown |
|-----------------|-------------|------------|--------|--------------|
| Max Sharpe      | {tearsheet.loc['Max Sharpe','Ann. Return %']}%       | {tearsheet.loc['Max Sharpe','Volatility %']}%      | {tearsheet.loc['Max Sharpe','Sharpe Ratio']}  | {tearsheet.loc['Max Sharpe','Max Drawdown %']}%       |
| Min Variance    | {tearsheet.loc['Min Variance','Ann. Return %']}%       | {tearsheet.loc['Min Variance','Volatility %']}%      | {tearsheet.loc['Min Variance','Sharpe Ratio']}  | {tearsheet.loc['Min Variance','Max Drawdown %']}%       |
| Risk Parity     | {tearsheet.loc['Risk Parity','Ann. Return %']}%       | {tearsheet.loc['Risk Parity','Volatility %']}%      | {tearsheet.loc['Risk Parity','Sharpe Ratio']}  | {tearsheet.loc['Risk Parity','Max Drawdown %']}%       |
| Equal Weight    | {tearsheet.loc['Equal Weight','Ann. Return %']}%       | {tearsheet.loc['Equal Weight','Volatility %']}%      | {tearsheet.loc['Equal Weight','Sharpe Ratio']}  | {tearsheet.loc['Equal Weight','Max Drawdown %']}%       |
| 60/40 Benchmark | {tearsheet.loc['60/40 Benchmark','Ann. Return %']}%       | {tearsheet.loc['60/40 Benchmark','Volatility %']}%      | {tearsheet.loc['60/40 Benchmark','Sharpe Ratio']}  | {tearsheet.loc['60/40 Benchmark','Max Drawdown %']}%       |

Best strategy ({best_strategy}) achieved a {sharpe_improv}% Sharpe improvement vs 60/40 benchmark.

## Notebooks

| Notebook | Description |
|----------|-------------|
| 01_Data_Pipeline | Fetch and clean 7-asset price data (2018-2024) |
| 02_Portfolio_Analytics | Risk metrics, VaR, Sharpe, drawdown analysis |
| 03_Optimization_Engine | Efficient frontier and optimal portfolio construction |
| 04_Backtesting_Engine | Walk-forward backtest vs 60/40 benchmark |
| 05_Final_Report | Master dashboard and performance tearsheet |

## Key Visualizations

- Efficient Frontier with 10,000 Monte Carlo portfolios
- Cumulative growth of $100,000 across all strategies
- Correlation heatmap across asset classes
- Rolling 12-month Sharpe ratio by strategy
- Value at Risk on $1,000,000 portfolio
- Annual returns heatmap (2018-2024)

## Tools and Libraries

Python | Pandas | NumPy | SciPy | Plotly | yFinance

## Author

Your Name | Financial Analyst
LinkedIn: linkedin.com/in/yourprofile
"""

print(readme)
print("")
print("Copy everything above into your GitHub README.md file")


# Multi-Asset Portfolio Optimizer

> Institutional-grade portfolio construction across 7 asset classes
> using Modern Portfolio Theory, Risk Parity, and Monte Carlo simulation.

## Project Overview

This project implements a complete quantitative portfolio management
framework — from data pipeline to backtesting — replicating the
methodology used by institutional asset managers.

## Asset Universe

| Ticker | Asset Class        |
|--------|--------------------|
| SPY    | US Equities        |
| EFA    | Intl Equities      |
| TLT    | Long-Term Bonds    |
| BND    | Aggregate Bonds    |
| GLD    | Gold               |
| GSG    | Commodities        |
| VNQ    | Real Estate (REITs)|

## Strategies Implemented

- Mean-Variance Optimization (Markowitz 1952)
- Maximum Sharpe Ratio Portfolio
- Minimum Variance Portfolio
- Risk Parity (Equal Risk Contribution)
- Walk-Forward Backtesting with Monthly Rebalancing

## Backtest Results (2019-2024)

| Strategy        | Ann. Return | Volatility | 